## Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

1) A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2) A function or coroutine to execute.

In [4]:
import os
from langchain.chat_models import init_chat_model


os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Why do parrots talk?")
response.content

'<think>\nOkay, so I need to figure out why parrots talk. Let\'s start by recalling what I know about parrots. They\'re birds, right? Some species like the African Grey, Macaws, and Budgies are known for talking. But why do they do that? Are they just imitating sounds, or is there a deeper reason?\n\nFirst, I remember that parrots are highly intelligent. They have large brains relative to their body size. Maybe their ability to mimic sounds is part of their intelligence. But why would they evolve to talk? In the wild, maybe they use vocalizations for communication within their flock. So, in captivity, they might imitate human speech to communicate with their human caregivers.\n\nI\'ve heard that parrots can learn words through training and repetition. So, their talking is a learned behavior. But there\'s also the aspect of social bonding. Parrots are social animals, and talking could help strengthen their bonds with humans. They might talk to get attention, request food, or just have i

In [ ]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the current weather for a given location."""
    return f"The current weather in {location} is sunny with a temperature of 25°C."


# binding a tool with the model
model_with_tool = model.bind_tools([get_weather])

In [6]:
response = model_with_tool.invoke("What's the weather like in Boston?")

for tool_call in response.tool_calls:
    # view tool calls made by the model
    print(tool_call)
    print(f"Tool name: {tool_call['name']}")
    print(f"Tool arguments: {tool_call['args']}")


{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': '7g70b20zq', 'type': 'tool_call'}
Tool name: get_weather
Tool arguments: {'location': 'Boston'}


### Tool execution loop

In [9]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tool.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tool.invoke(messages)
print(final_response.text)
print(messages)

The current weather in Boston is sunny with a temperature of 25°C.
[{'role': 'user', 'content': "What's the weather in Boston?"}, AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. I need to use the get_weather function. Let me check the function parameters. The required parameter is location, which should be a string. So I\'ll call get_weather with location set to "Boston". Make sure the JSON is correctly formatted with the name and arguments. No other parameters are needed here. Just pass "Boston" as the location.\n', 'tool_calls': [{'id': '4spwfa6s4', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 106, 'prompt_tokens': 155, 'total_tokens': 261, 'completion_time': 0.165301384, 'completion_tokens_details': {'reasoning_tokens': 82}, 'prompt_time': 0.006034417, 'prompt_tokens_details': None, 'queue_time': 0.049083473, 